# The Gymnasium interface of a planning problem

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/teaching/courses/udes_gro860/gymnasium_interface.ipynb)

[Gymnasium](https://gymnasium.farama.org) is the standard Python interface for reinforcement-learning environments. Almost every RL library trains against it. The contract is small: two spaces and two methods.

This notebook reads that contract in optimal-control terms, writes an environment by hand, obtains the same environment from minilink, trains through [Stable-Baselines3](https://stable-baselines3.readthedocs.io) (including a planar drone that learns to fly), and then solves the same drone task with minilink's native planner — a compiled loop, much faster.

This page uses the [minilink](https://github.com/alx87grd/minilink) toolbox.


In [ ]:
# Local: minilink already installed. Colab: clone + path + RL libraries.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q gymnasium stable-baselines3")

In [ ]:
import gymnasium as gym
import numpy as np
from gymnasium import spaces
from gymnasium.utils.env_checker import check_env
from stable_baselines3 import PPO

from minilink import (
    CostFunction,
    Drone2D,
    Gaussian,
    Pendulum,
    ReinforcementLearningPlanner,
    StochasticPlanningProblem,
    Uniform,
)
from minilink.interfaces.gymnasium import SB3Controller, Sys2Gym

## 1. How Gymnasium works

| Gymnasium | Optimal control |
| --- | --- |
| `observation_space` | the set of observations $y$ |
| `action_space` | the admissible inputs $U$ |
| `reset(seed)` → `(y, info)` | draw $x_0$ |
| `step(u)` → `(y, r, terminated, truncated, info)` | one control period |

One `step` is four sub-steps:

1. **Dynamics:** $x_{k+1} = f(x_k, u_k)$ over $\Delta t$, the input held.
2. **Reward:** $r_k = -g(x_k, u_k)\,\Delta t$. Gymnasium maximizes reward, so the sign flips the cost.
3. **Observation:** $y_{k+1} = h(x_{k+1})$.
4. **Episode end:** *terminated* when nothing is left to count (a goal, a crash, a finite horizon after its terminal cost); *truncated* when the episode is cut short but the future still has value (a time limit, leaving the training box). A learning algorithm bootstraps a truncated episode with the value of its last state, and not a terminated one.

The class holds the state between calls: `reset` sets it, `step` advances it.


## 2. The planning problem

Gymnasium is an *interface*, not a problem statement. The objects that define the task are still those of optimal control:

- the plant $\dot x = f(x,u)$ — here a torque-limited pendulum ($\theta = 0$ hanging, $\theta = \pi$ upright);
- a running cost $g(x,u)$ that is zero upright and two hanging, and $h = 0$;
- the admissible torque $U$ and a box on $x$;
- a start distribution for $x_0$;
- an infinite horizon, so an episode will be *truncated* by a time limit, not *terminated* by a goal.

The next cell writes that once, as a `StochasticPlanningProblem`. That object is what we translate into a Gymnasium environment, **two ways**:

1. **by hand** — a `gym.Env` whose `reset` / `step` copy $f$, $g$, and the start law;
2. **the minilink shortcut** — `Sys2Gym.from_problem(problem)`, the same mapping in one call.

Later sections do not change the math; they only change who implements the four sub-steps of §1.


In [ ]:
# The math: plant, cost, start law — not a Gym env yet

TORQUE = 4.0  # Nm, below m g l = 9.81 Nm
DT = 0.05

plant = Pendulum()
plant.inputs["u"].lower_bound = np.array([-TORQUE])
plant.inputs["u"].upper_bound = np.array([TORQUE])
plant.state.lower_bound = np.array([-4 * np.pi, -20.0])
plant.state.upper_bound = np.array([4 * np.pi, 20.0])


class SwingUpCost(CostFunction):
    def g(self, x, u, t=0.0, params=None):
        theta, dtheta = x
        return (1.0 + np.cos(theta)) + 0.01 * dtheta**2 + 0.01 * u[0] ** 2

    def h(self, x, t=0.0, params=None):
        return 0.0


cost = SwingUpCost()
problem = StochasticPlanningProblem(
    plant,
    cost=cost,
    tf=np.inf,
    x0_distribution=Uniform([-np.pi, -1.0], [np.pi, 1.0]),
)

## 3. Translation 1 — by hand

`PendulumSwingUpEnv` is the Gymnasium view of the problem above: the observation and action spaces are the bounds on $x$ and $u$, `reset` draws from the same uniform start, and `step` is one Euler period of $f$ with reward $-g\,\Delta t$. Read it against the four sub-steps of §1.


In [ ]:
class PendulumSwingUpEnv(gym.Env):
    """The swing-up task written against the Gymnasium interface by hand."""

    def __init__(self, dt=DT, tf=10.0):
        # One env step = one control period of length dt; tf is the time-limit truncation
        self.dt, self.tf = dt, tf

        # Observation space = the set of y. Here y = x (full state), so the plant's state box
        x_lb, x_ub = plant.state.lower_bound, plant.state.upper_bound
        self.observation_space = spaces.Box(x_lb.astype(np.float32), x_ub.astype(np.float32))

        # Action space = the admissible inputs U (the torque box)
        u_lb, u_ub = plant.inputs["u"].lower_bound, plant.inputs["u"].upper_bound
        self.action_space = spaces.Box(u_lb.astype(np.float32), u_ub.astype(np.float32))

    def reset(self, seed=None, options=None):
        # Seed Gymnasium's RNG (self.np_random); required by the contract
        super().reset(seed=seed)

        # Draw x0 from the same uniform as the planning problem; start the clock
        self.x = self.np_random.uniform([-np.pi, -1.0], [np.pi, 1.0])
        self.t = 0.0

        # Return (y, info). y is float32 to match observation_space; info is optional extras
        return self.x.astype(np.float32), {}

    def step(self, u):
        x, t, dt = self.x, self.t, self.dt

        # Enforce U: clip the action to the torque box, hold it over dt (zero-order hold)
        u = np.clip(np.asarray(u, dtype=float), self.action_space.low, self.action_space.high)

        # 1. Dynamics: x' = x + f(x, u) dt  (Euler; the plant supplies f)
        x_next = x + plant.f(x, u, t) * dt

        # 2. Reward: r = -g(x, u) dt  (Gymnasium maximizes; the cost is minimized)
        reward = -float(cost.g(x, u, t)) * dt

        # 3. Observation: y = h(x') = x'  (state feedback)
        y = x_next.astype(np.float32)

        # 4. Episode end
        terminated = False  # infinite horizon: no goal / crash that ends the cost
        outside = np.any(x_next < self.observation_space.low) or np.any(
            x_next > self.observation_space.high
        )
        truncated = bool(t + dt > self.tf or outside)  # time limit or leaving the box

        # The class holds (x, t) between calls
        self.x, self.t = x_next, t + dt
        return y, reward, terminated, truncated, {}  # info unused


Gymnasium ships a validator of the contract. A random policy then runs one episode; its return is minus its cost.


In [ ]:
env = PendulumSwingUpEnv()
check_env(env)

y, info = env.reset(seed=0)
print("reset:", y, info)

episode_return, k = 0.0, 0
terminated = truncated = False
while not (terminated or truncated):
    y, r, terminated, truncated, info = env.step(env.action_space.sample())
    episode_return += r
    k += 1
    print(k, y, r, terminated, truncated)
k, -episode_return

## 4. Translation 2 — the minilink shortcut

`Sys2Gym.from_problem` is the same translation, written once in the library: `reset` draws from the problem's start distribution, the reward is the running cost, and the horizon rules decide *terminated* or *truncated*. With `integrator="euler"` it is the class above, step for step.


In [ ]:
gym_env = Sys2Gym.from_problem(
    problem, dt=DT, integrator="euler", compile_backend="numpy"
)
check_env(gym_env)
gym_env.observation_space, gym_env.action_space

In [ ]:
y, info = gym_env.reset(seed=0)
print("reset:", y, info)

episode_return, k = 0.0, 0
terminated = truncated = False
while not (terminated or truncated):
    y, r, terminated, truncated, info = gym_env.step(gym_env.action_space.sample())
    episode_return += r
    k += 1
    print(k, y, r, terminated, truncated)
k, -episode_return

## 5. Learn to fly with Stable-Baselines3

[Stable-Baselines3](https://stable-baselines3.readthedocs.io) trains any Gymnasium environment. `PPO("MlpPolicy", env)` is a neural policy $\pi_\theta(y)$. After training, its native call is `predict`:
$$u = \pi_\theta(y).$$
We use that policy **two ways**, the same pattern as the environment:

1. **by hand** — `model.predict` inside the `reset` / `step` loop;
2. **the minilink shortcut** — `SB3Controller` wraps `predict` as a block $u=\pi_\theta(x)$, so `ctl @ plant` closes the loop like LQR.

The task is a planar drone: $x = [x,\; y,\; \theta,\; v_x,\; v_y,\; \omega]$, $u = [T_1,\; T_2]$ normalized so $u=0$ hovers. First the plant and the Gym env, then we train, then the two translations.


In [ ]:
class NormalizedDrone2D(Drone2D):
    """Planar drone with thrust inputs normalized between -1 and 1."""

    def __init__(self):
        super().__init__()
        self.params["mass"] = 1.0
        self.params["inertia"] = 0.1
        self.inputs["u"].lower_bound = np.array([-1.0, -1.0])
        self.inputs["u"].upper_bound = np.array([+1.0, +1.0])
        self.inputs["u"].units = ["%", "%"]
        self.weight = self.params["gravity"] * self.params["mass"]
        self.thrust2weight = 1.2
        self.state.upper_bound = np.array([10, 10, 2 * np.pi, 10, 10, 10])
        self.state.lower_bound = -self.state.upper_bound

    def thrust(self, u):
        return self.weight * ((self.thrust2weight - 1.0) * u + np.array([0.5, 0.5]))

    def f(self, x, u, t=0.0, params=None):
        return super().f(x, self.thrust(u), t, params)

    def get_dynamic_geometry(self, x, u, t=0, params=None):
        return super().get_dynamic_geometry(x, self.thrust(u), t, params)


class HoverCost(CostFunction):
    Q = np.diag([1.0, 1.0, 6.0, 0.1, 0.1, 0.1])
    R = np.diag([0.001, 0.001])

    def g(self, x, u, t=0.0, params=None):
        Q, R = self.Q, self.R
        return x @ Q @ x + u @ R @ u

    def h(self, x, t=0.0, params=None):
        return 0.0


drone = NormalizedDrone2D()
drone.x0 = np.zeros(6)
hover = HoverCost()
drone_env = Sys2Gym(drone, hover, dt=0.05)
drone_env.reset_mode = "gaussian"
drone_env.x0_std = np.array([5.0, 5.0, 1.0, 1.0, 1.0, 0.2])

In [ ]:
nn = PPO("MlpPolicy", drone_env, verbose=1)

Training updates $\theta$ from sampled trajectories. Start short, then train more if the law has not settled.


In [ ]:
nn.learn(20_000)

In [ ]:
# nn.learn(200_000)

### Translation 1 — by hand (`predict`)

`predict(y)` returns $u$, which we feed to `step`. `deterministic=True` takes the mean action (no exploration).


In [ ]:
y, info = drone_env.reset(seed=0)
print("reset:", y, info)

episode_return, k = 0.0, 0
terminated = truncated = False
while not (terminated or truncated):
    u, _ = nn.predict(y, deterministic=True)
    y, r, terminated, truncated, info = drone_env.step(u)
    episode_return += r
    k += 1
    print(k, y, u, r, terminated, truncated)
k, -episode_return

### Translation 2 — the minilink shortcut (`SB3Controller`)

`SB3Controller` is the same `predict` as a minilink block: $u=\pi_\theta(x)$ on the state port, so `ctl @ plant` is the closed loop.


In [ ]:
ppo_ctl = SB3Controller(nn, sys=drone)

In [ ]:
ppo_ctl.plot_control_law(x_axis=2, y_axis=5, u_axis=0)  # T1 vs (theta, omega)

In [ ]:
ppo_ctl.plot_control_law(x_axis=2, y_axis=5, u_axis=1)  # T2 vs (theta, omega)

In [ ]:
drone.x0 = np.array([-1.0, -2.0, 1.0, 0.0, 0.0, 0.0])
cl_sb3 = ppo_ctl @ drone
traj_sb3 = cl_sb3.compute_trajectory(tf=10.0, dt=0.01)

In [ ]:
cl_sb3.plot_trajectory(traj_sb3)

In [ ]:
cl_sb3.animate(traj_sb3)

## 6. The same task, native

No Gymnasium loop: the planner compiles the plant and PPO-updates a neural law on a batch of trajectories. Same $f$, same $g$, same start distribution — typically an order of magnitude more steps per second than stepping `Sys2Gym` from Python.


In [ ]:
drone.x0 = np.array([-1.0, -2.0, 1.0, 0.0, 0.0, 0.0])
problem = StochasticPlanningProblem(
    drone, cost=hover, tf=np.inf,
    x0_distribution=Gaussian(np.zeros(6), [5.0, 5.0, 1.0, 1.0, 1.0, 0.2]),
)
native_ctl = ReinforcementLearningPlanner(problem, dt=0.05).solve(timesteps=200_000).policy
cl = native_ctl @ drone
traj = cl.compute_trajectory(tf=10.0, dt=0.01)

In [ ]:
native_ctl.plot_control_law(x_axis=2, y_axis=5, u_axis=0)

In [ ]:
cl.plot_trajectory(traj)

In [ ]:
cl.animate(traj)